In [37]:
import sys
sys.path.insert(0, './hy3dshape')
sys.path.insert(0, './hy3dpaint')

import os
import logging
from PIL import Image
from hy3dshape.rembg import BackgroundRemover
from hy3dshape.pipelines import Hunyuan3DDiTFlowMatchingPipeline


from textureGenPipeline import Hunyuan3DPaintPipeline, Hunyuan3DPaintConfig

try:
    from torchvision_fix import apply_fix
    apply_fix()
except ImportError:
    print("Warning: torchvision_fix module not found, proceeding without compatibility fix")                                      
except Exception as e:
    print(f"Warning: Failed to apply torchvision fix: {e}")

Torchvision version: 0.20.1+cu124
torchvision.transforms.functional_tensor is available


In [29]:
# shape
model_path = 'tencent/Hunyuan3D-2.1'
mini_model_path='tencent/Hunyuan3D-2mini'

In [30]:
def get_logger(name):
    logger = logging.getLogger(name)
    logger.setLevel(logging.INFO)

    console_handler = logging.StreamHandler()
    console_handler.setLevel(logging.INFO)

    formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
    console_handler.setFormatter(formatter)
    logger.addHandler(console_handler)
    return logger


In [35]:
original_model_path = mini_model_path
subfolder='hunyuan3d-dit-v2-1'
logger = get_logger('hy3dgen.shapgen')

    # try local path
base_dir = os.environ.get('HY3DGEN_MODELS', '~/.cache/hy3dgen')
model_fld = os.path.expanduser(os.path.join(base_dir, model_path))
model_path = os.path.expanduser(os.path.join(base_dir, model_path, subfolder))
logger.info(f'Try to load model from local path: {model_path}')
if not os.path.exists(model_path):
    logger.info('Model path not exists, try to download from huggingface')
    try:
        from huggingface_hub import snapshot_download
        # 只下载指定子目录
        

        path = snapshot_download(
            repo_id=original_model_path,
            allow_patterns=[f"{subfolder}/*"],  # 关键修改：模式匹配子文件夹
            local_dir=model_fld 
        )
        model_path = os.path.join(path, subfolder)  # 保持路径拼接逻辑不变
        logger.info(f'End {model_path}')
    except ImportError:
        logger.warning(
            "You need to install HuggingFace Hub to load models from the hub."
        )
        raise RuntimeError(f"Model path {model_path} not found")
    except Exception as e:
        raise e

2025-11-27 10:16:44,898 - hy3dgen.shapgen - INFO - Try to load model from local path: /dcs/pg24/u5745134/.cache/hy3dgen/tencent/Hunyuan3D-2.1/hunyuan3d-dit-v2-1/hunyuan3d-dit-v2-1
2025-11-27 10:16:44,898 - hy3dgen.shapgen - INFO - Try to load model from local path: /dcs/pg24/u5745134/.cache/hy3dgen/tencent/Hunyuan3D-2.1/hunyuan3d-dit-v2-1/hunyuan3d-dit-v2-1
2025-11-27 10:16:44,898 - hy3dgen.shapgen - INFO - Try to load model from local path: /dcs/pg24/u5745134/.cache/hy3dgen/tencent/Hunyuan3D-2.1/hunyuan3d-dit-v2-1/hunyuan3d-dit-v2-1
2025-11-27 10:16:44,898 - hy3dgen.shapgen - INFO - Try to load model from local path: /dcs/pg24/u5745134/.cache/hy3dgen/tencent/Hunyuan3D-2.1/hunyuan3d-dit-v2-1/hunyuan3d-dit-v2-1
2025-11-27 10:16:44,898 - hy3dgen.shapgen - INFO - Try to load model from local path: /dcs/pg24/u5745134/.cache/hy3dgen/tencent/Hunyuan3D-2.1/hunyuan3d-dit-v2-1/hunyuan3d-dit-v2-1
2025-11-27 10:16:44,898 - hy3dgen.shapgen - INFO - Try to load model from local path: /dcs/pg24/u574

In [33]:


pipeline_shapegen = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained(mini_model_path,device='cpu',num_inference_steps=5)

image_path = '/dcs/pg24/u5745134/Desktop/dev/Hunyuan3D-2.1/assets/demo.png'
image = Image.open(image_path).convert("RGBA")
if image.mode == 'RGB':
    rembg = BackgroundRemover()
    image = rembg(image)

mesh = pipeline_shapegen(image=image)[0]

save_dir = '/dcs/pg24/u5745134/Desktop/dev/gen_example_meshes'
os.makedirs(save_dir, exist_ok=True)

mesh.export(f'{save_dir}/demo.glb')

2025-11-27 10:15:02,869 - hy3dgen.shapgen - INFO - Try to load model from local path: /dcs/pg24/u5745134/.cache/hy3dgen/tencent/Hunyuan3D-2mini/hunyuan3d-dit-v2-1
2025-11-27 10:15:02,869 - hy3dgen.shapgen - INFO - Try to load model from local path: /dcs/pg24/u5745134/.cache/hy3dgen/tencent/Hunyuan3D-2mini/hunyuan3d-dit-v2-1
2025-11-27 10:15:02,869 - hy3dgen.shapgen - INFO - Try to load model from local path: /dcs/pg24/u5745134/.cache/hy3dgen/tencent/Hunyuan3D-2mini/hunyuan3d-dit-v2-1
2025-11-27 10:15:02,869 - hy3dgen.shapgen - INFO - Try to load model from local path: /dcs/pg24/u5745134/.cache/hy3dgen/tencent/Hunyuan3D-2mini/hunyuan3d-dit-v2-1
2025-11-27 10:15:02,869 - hy3dgen.shapgen - INFO - Try to load model from local path: /dcs/pg24/u5745134/.cache/hy3dgen/tencent/Hunyuan3D-2mini/hunyuan3d-dit-v2-1
2025-11-27 10:15:02,871 - hy3dgen.shapgen - INFO - Model path not exists, try to download from huggingface
2025-11-27 10:15:02,871 - hy3dgen.shapgen - INFO - Model path not exists, try 

FileNotFoundError: Model path tencent/Hunyuan3D-2mini not found

In [32]:
!which jupyter-notebook

/dcs/large/u5745134/hunyuanenv/bin/jupyter-notebook


In [43]:
import time
import hy3dshape
from hy3dshape.rembg import BackgroundRemover
from hy3dshape.pipelines import Hunyuan3DDiTFlowMatchingPipeline
#image_path = 'assets/demo.png'
#image = Image.open(image_path).convert("RGBA")
#if image.mode == 'RGB':
#    rembg = BackgroundRemover()
#    image = rembg(image)

pipeline = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained(
    'tencent/Hunyuan3D-2',
    subfolder='hunyuan3d-dit-v2-0',
    #variant='fp16',
    device ="cpu"
)

captions = {
    'chairs': {
        'decimal': "A slat back wooden chair of 0,9 meters with 4 legs and a seat (height: 0,45m, width: 0,4m), It has 5 vertical backrest slats spaced at 0,008m",
        'fraction': "A slat back wooden chair of 9/10 meters with four legs and a seat (height: 9/2m, width: 2/5m), It has five vertical backrest slats spaced at 1/125 of a meter",
        'words': "A slat back wooden chair of zero point nine meters with four legs and a seat (height: zero point fourty-five meters, width: zero point four meters). It has five vertical backrest slats spaced at zero point zero zero eight meters.",
    },
    'bookcases': {
        'decimal': "A black rectangular bookcase (1.8m high, 0.9m wide, 0.3m deep),Include 6 evenly spaced shelves (0.28m apart, 0.02m thick), 1 wide drawer at the bottom",
        'fraction': "A black rectangular bookcase (9/5m high, 9/10m wide, 3/10m deep) Include six evenly spaced shelves (7/25m apart, 1/50m thick), one wide drawer at bottom",
        'words': "A black rectangular bookcase (one point eight meters high, zero point nine meters wide, zero point three meters deep), includes six evenly spaced shelves (zero point two eight meters apart, zero point zero two meters thick), and one wide drawer at the bottom",
    },
    'tables': {
        'decimal': "А round table with a tabletop (diameter: 0,8m), supported by 3 slender legs (height: 0,4m), The legs connect to a circular gold base ring.",
        'fraction': "А round table with a tabletop (diameter: 4/5m), supported by three slender legs (height: 2/5m). The legs connect to a circular gold base ring.",
        'words': "A round table with a tabletop (diameter: zero point eight meters), supported by three slender legs (height: zero point four meters). The legs connect to a circular gold base ring.",
    }
    }

start_time = time.time()
mesh = pipeline(caption = captions["chairs"]["words"],
                num_inference_steps=50,
                octree_resolution=380,
                num_chunks=20000,
                generator=torch.manual_seed(12345),
                output_type='trimesh'
                )[0]
print("--- %s seconds ---" % (time.time() - start_time))
mesh.export(f'chair_dec.glb')

2025-11-27 10:54:06,364 - hy3dgen.shapgen - INFO - Try to load model from local path: /dcs/pg24/u5745134/.cache/hy3dgen/tencent/Hunyuan3D-2/hunyuan3d-dit-v2-0
2025-11-27 10:54:06,364 - hy3dgen.shapgen - INFO - Try to load model from local path: /dcs/pg24/u5745134/.cache/hy3dgen/tencent/Hunyuan3D-2/hunyuan3d-dit-v2-0
2025-11-27 10:54:06,364 - hy3dgen.shapgen - INFO - Try to load model from local path: /dcs/pg24/u5745134/.cache/hy3dgen/tencent/Hunyuan3D-2/hunyuan3d-dit-v2-0
2025-11-27 10:54:06,364 - hy3dgen.shapgen - INFO - Try to load model from local path: /dcs/pg24/u5745134/.cache/hy3dgen/tencent/Hunyuan3D-2/hunyuan3d-dit-v2-0
2025-11-27 10:54:06,364 - hy3dgen.shapgen - INFO - Try to load model from local path: /dcs/pg24/u5745134/.cache/hy3dgen/tencent/Hunyuan3D-2/hunyuan3d-dit-v2-0
2025-11-27 10:54:06,364 - hy3dgen.shapgen - INFO - Try to load model from local path: /dcs/pg24/u5745134/.cache/hy3dgen/tencent/Hunyuan3D-2/hunyuan3d-dit-v2-0
2025-11-27 10:54:06,364 - hy3dgen.shapgen - IN

ModuleNotFoundError: No module named 'hy3dgen'